In [3]:
import duckdb

duckdb.sql("""
    INSTALL httpfs;
    LOAD httpfs;
    SET s3_endpoint='localhost:9000';
    SET s3_access_key_id='minioadmin';
    SET s3_secret_access_key='minioadmin';
    SET s3_use_ssl=false;
    SET s3_url_style='path';
""")

In [58]:
duckdb.sql(f"""
        SELECT *
        FROM read_csv_auto('s3://bronze/users_data.csv', strict_mode=false)
        LIMIT 1000
        """).show()

┌───────┬─────────────┬────────────────┬────────────┬─────────────┬─────────┬──────────────────────────┬──────────┬───────────┬───────────────────┬───────────────┬────────────┬──────────────┬──────────────────┐
│  id   │ current_age │ retirement_age │ birth_year │ birth_month │ gender  │         address          │ latitude │ longitude │ per_capita_income │ yearly_income │ total_debt │ credit_score │ num_credit_cards │
│ int64 │    int64    │     int64      │   int64    │    int64    │ varchar │         varchar          │  double  │  double   │      varchar      │    varchar    │  varchar   │    int64     │      int64       │
├───────┼─────────────┼────────────────┼────────────┼─────────────┼─────────┼──────────────────────────┼──────────┼───────────┼───────────────────┼───────────────┼────────────┼──────────────┼──────────────────┤
│   825 │          53 │             66 │       1966 │          11 │ Female  │ 462 Rose Lane            │    34.15 │   -117.76 │ $29278            │ $59696  

In [7]:

duckdb.sql(f"""
        SELECT COUNT(*) 
        FROM read_csv_auto('s3://bronze/users_data.csv', strict_mode=false)
        """).show()



duckdb.sql(f"""
        SELECT COUNT(*) 
        FROM read_csv_auto('s3://bronze/users_data.csv', ignore_errors=true)
        """).show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│         2000 │
└──────────────┘

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│         2000 │
└──────────────┘



In [12]:

duckdb.sql(f"""
        DESC FROM read_csv_auto('s3://bronze/users_data.csv', strict_mode=false)
        """).show()

┌───────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name    │ column_type │  null   │   key   │ default │  extra  │
│      varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ id                │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ current_age       │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ retirement_age    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ birth_year        │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ birth_month       │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gender            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ address           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ latitude          │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ longitude         │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ per_capita

In [ ]:
duckdb.sql(f"""
    SELECT 'current_age - MAX' TIPO, MAX(current_age) VALOR
    FROM read_csv_auto('s3://bronze/users_data.csv')
    UNION ALL
    SELECT 'current_age - MIN',   MIN(current_age)
    FROM read_csv_auto('s3://bronze/users_data.csv')
    UNION ALL
    SELECT 'current_age - NULOS',  COUNT(*)
    FROM read_csv_auto('s3://bronze/users_data.csv')
    WHERE current_age IS NULL
    UNION ALL
    SELECT 'retirement_age - MAX',  MAX(retirement_age)
    FROM read_csv_auto('s3://bronze/users_data.csv')
    UNION ALL
    SELECT 'retirement_age - MIN',  MIN(retirement_age)
    FROM read_csv_auto('s3://bronze/users_data.csv')
    UNION ALL
    SELECT 'retirement_age - NULOS',  COUNT(*)
    FROM read_csv_auto('s3://bronze/users_data.csv')
    WHERE retirement_age IS NULL
""").show()

┌────────────────────────┬───────┐
│          TIPO          │ VALOR │
│        varchar         │ int64 │
├────────────────────────┼───────┤
│ current_age - MAX      │   101 │
│ current_age - MIN      │    18 │
│ current_age - NULOS    │     0 │
│ retirement_age - MAX   │    79 │
│ retirement_age - MIN   │    50 │
│ retirement_age - NULOS │     0 │
└────────────────────────┴───────┘



In [76]:
duckdb.sql(f"""
    SELECT
        MAX(birth_year)               birth_year_max,
        MIN(birth_year)               birth_year_min,
        COUNT(*) - COUNT(birth_year)  birth_year_nulos,
        MAX(birth_month)              birth_month_max,
        MIN(birth_month)              birth_month_min,
        COUNT(*) - COUNT(birth_month) birth_month_nulos
    FROM read_csv_auto('s3://bronze/users_data.csv')
""").show()

┌────────────────┬────────────────┬──────────────────┬─────────────────┬─────────────────┬───────────────────┐
│ birth_year_max │ birth_year_min │ birth_year_nulos │ birth_month_max │ birth_month_min │ birth_month_nulos │
│     int64      │     int64      │      int64       │      int64      │      int64      │       int64       │
├────────────────┼────────────────┼──────────────────┼─────────────────┼─────────────────┼───────────────────┤
│           2002 │           1918 │                0 │              12 │               1 │                 0 │
└────────────────┴────────────────┴──────────────────┴─────────────────┴─────────────────┴───────────────────┘



In [42]:
duckdb.sql(f"""
        SELECT gender, COUNT(*) QTD
        FROM read_csv_auto('s3://bronze/users_data.csv')
        GROUP BY gender
        ORDER BY 2 DESC 
        """).show()

┌─────────┬───────┐
│ gender  │  QTD  │
│ varchar │ int64 │
├─────────┼───────┤
│ Female  │  1016 │
│ Male    │   984 │
└─────────┴───────┘



In [59]:
duckdb.sql(f"""
        SELECT COUNT(DISTINCT address), COUNT(address) 
        FROM read_csv_auto('s3://bronze/users_data.csv')
""").show()

duckdb.sql(f"""
        SELECT address, COUNT(*) QTD
        FROM read_csv_auto('s3://bronze/users_data.csv')
        GROUP BY address
        HAVING QTD > 1 
        ORDER BY 2 DESC 
""").show()

duckdb.sql(f"""
        SELECT COUNT(address), count(TRIM(UPPER(address))) 
        FROM read_csv_auto('s3://bronze/users_data.csv')
""").show()


duckdb.sql(f"""
        SELECT COUNT(DISTINCT regexp_replace(address,'[0-9]+','')) ENDERECO
        FROM read_csv_auto('s3://bronze/users_data.csv')
""").show()



┌─────────────────────────┬────────────────┐
│ count(DISTINCT address) │ count(address) │
│          int64          │     int64      │
├─────────────────────────┼────────────────┤
│                    1999 │           2000 │
└─────────────────────────┴────────────────┘

┌─────────────────────┬───────┐
│       address       │  QTD  │
│       varchar       │ int64 │
├─────────────────────┼───────┤
│ 506 Washington Lane │     2 │
└─────────────────────┴───────┘

┌────────────────┬────────────────────────────────────┐
│ count(address) │ count(main."trim"(upper(address))) │
│     int64      │               int64                │
├────────────────┼────────────────────────────────────┤
│           2000 │                               2000 │
└────────────────┴────────────────────────────────────┘

┌──────────┐
│ ENDERECO │
│  int64   │
├──────────┤
│      290 │
└──────────┘



In [63]:
duckdb.sql(f"""
        SELECT MAX(latitude), MIN(longitude) 
        FROM read_csv_auto('s3://bronze/users_data.csv')
""").show()

duckdb.sql(f"""
        SELECT COUNT(*)
        FROM read_csv_auto('s3://bronze/users_data.csv')
        WHERE latitude IS NULL OR longitude IS NULL
""").show()

┌───────────────┬────────────────┐
│ max(latitude) │ min(longitude) │
│    double     │     double     │
├───────────────┼────────────────┤
│          61.2 │        -159.41 │
└───────────────┴────────────────┘

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│            0 │
└──────────────┘



In [ ]:

duckdb.sql(f"""
        SELECT 
                MAX(per_capita_income) MAX_per_capita_income, MIN(per_capita_income) MIN_per_capita_income, 
                MAX(yearly_income) MAX_yearly_income, MIN(yearly_income) MIN_yearly_income,
                MAX(total_debt) MAX_total_dept, MIN(total_debt) MIN_total_dept
        FROM read_csv_auto('s3://bronze/users_data.csv')       
""").show()

duckdb.sql(f"""
        SELECT  
                MAX(credit_score) MAX_credit_score, MIN(credit_score) MIN_credit_score,
                MAX(num_credit_cards ) MAX_num_credit_cards, MIN(num_credit_cards ) MIN_num_credit_cards 
        FROM read_csv_auto('s3://bronze/users_data.csv')      
""").show()

┌────────────────┬────────────────┬───────────────────┬───────────────────┬────────────────┬────────────────┐
│ MAX_PER_CAPITA │ MIN_PER_CAPITA │ MAX_YEARLY_INCOME │ MIN_YEARLY_INCOME │ MAX_total_dept │ MIN_total_dept │
│    varchar     │    varchar     │      varchar      │      varchar      │    varchar     │    varchar     │
├────────────────┼────────────────┼───────────────────┼───────────────────┼────────────────┼────────────────┤
│ $9995          │ $0             │ $99883            │ $1                │ $99840         │ $0             │
└────────────────┴────────────────┴───────────────────┴───────────────────┴────────────────┴────────────────┘

┌──────────────────┬──────────────────┬──────────────────────┬──────────────────────┐
│ MAX_credit_score │ MIN_credit_score │ MAX_num_credit_cards │ MIN_num_credit_cards │
│      int64       │      int64       │        int64         │        int64         │
├──────────────────┼──────────────────┼──────────────────────┼───────────────────

In [86]:

duckdb.sql(f"""
        SELECT COUNT(*)  
        FROM read_csv_auto('s3://bronze/users_data.csv')   
        WHERE per_capita_income = '$0' OR yearly_income = '$0' OR total_debt = '$0'
""").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│          113 │
└──────────────┘

